TypedDict 是 Python 3.8+ 引入的一种类型提示工具，即带有类型声明的字典结构。适合需要快速定义字典结构且无需 Pydantic 重量级功能的场景。

普通 dict 没有类型信息。

TypedDict 可以进一步说明：
- 这个字典应该有哪些字段
- 每个字段的类型是什么
- TypedDict 主要是类型声明，不是运行时强校验器。

In [4]:
from typing_extensions import TypedDict

class MovieDict(TypedDict):
    title: str
    year: int
    director: str
    rating: float

#故意，让new_title ，而不是title
movie: MovieDict = {
    "new_title": "盗梦空间",
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 8.8,
}
print(movie)

{'new_title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}


Annotated 用来在“类型”之外，再附加一些额外信息，即元数据。类似于 Pydantic 的 Field 。

In [7]:
from langchain.chat_models import init_chat_model
from typing import TypedDict, Annotated, List
from dotenv import load_dotenv
from rich import print as rprint
import os

#优先加载配置
load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# 使用TypedDict定义嵌套结构
class Actor(TypedDict):
    """演员情况"""
    name: Annotated[str, "演员姓名"]
    role: Annotated[str, "饰演的角色"]

class MovieTypedDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, "电影的正式名称，例如《盗梦空间》"]
    year: Annotated[int, "电影的公映年份，使用四位数字表示"]
    director: Annotated[str, "电影导演的全名"]
    cast: Annotated[List[Actor], "演员列表"] # 嵌套列表定义
    rating: Annotated[float, "电影在10分制下的评分，可包含一位小数"]

#设置模型结构化输出
structured_llm = model_openai.with_structured_output(MovieTypedDict)

response = structured_llm.invoke("给我介绍下电影《星际穿越》")

rprint(response)
print(type(response))


{
    'title': '星际穿越',
    'year': 2014,
    'director': '克里斯托弗·诺兰',
    'cast': [
        {'name': '马修·麦康纳', 'role': '库珀'},
        {'name': '安妮·海瑟薇', 'role': '布兰德博士'},
        {'name': '杰西卡·查斯坦', 'role': '墨菲（成年）'},
        {'name': '迈克尔·凯恩', 'role': '米勒教授'},
        {'name': '乔什·卢卡斯', 'role': '多伊尔'}
    ],
    'rating': 9.4
}

<class 'dict'>


...的使用

说明：

... 是Python的字面量，等价于 Ellipsis ，可以理解为占位符。下游框架（如LangChain）可以对 ... 作定制化处理，如LangChain中Annotated的 ... 表示当前字段是必须存在的，不可省略，用来指示模型的输出。

In [10]:
from langchain.chat_models import init_chat_model
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from rich import print as rprint
import os

#优先加载配置
load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

class MovieTypedDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, Ellipsis, "导演"]
    # 如果是Ellipsis或者 ...,那么就必须要有rating。当前大模型厂商给出的 -1
    rating: Annotated[float, "电影评分，满分十分"]

#设置模型结构化输出
structured_llm = model_openai.with_structured_output(MovieTypedDict)

response = structured_llm.invoke("根据这段话抽取盗梦空间的信息，不包含的信息可以留空：盗梦空间在2010年上映，导演是克里斯托弗·诺兰。")

rprint(response)


{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰'}